# Figure: dati e domanda OD

Notebook per aggiornare le figure descrittive della sezione dati del paper. Usa le matrici OD e l'attivita telefonica gia presenti nella repository. Gli output sono salvati in `article/results/figures/data/` in formato PDF e PNG.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def find_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'article').exists() and (candidate / 'MCF' / 'data').exists():
            return candidate
    raise FileNotFoundError('Eseguire il notebook dentro la repository covid_hep.')

ROOT = find_root()
RAW = ROOT / 'MCF' / 'data' / 'raw_inputs'
DER = ROOT / 'MCF' / 'data' / 'derived'
OUT = ROOT / 'article' / 'results' / 'figures' / 'data'
OUT.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', context='paper', font_scale=1.15)
COLORS = {'Weekdays': '#146C94', 'Holidays': '#F28E2B'}

def save_figure(fig, name):
    for ext in ('pdf', 'png'):
        fig.savefig(OUT / f'{name}.{ext}', dpi=300, bbox_inches='tight')
    plt.close(fig)

print(f'Output: {OUT}')

Output: /mnt/c/Users/belli/OneDrive/Desktop/code/covid_hep/article/results/figures/data


## Ricostruzione opzionale delle tabelle OD

I CSV OD completi hanno intestazioni multiple e sono voluminosi. Di default le figure leggono le tabelle derivate gia prodotte nella repository; impostare `REBUILD_DERIVED = True` per rigenerarle dai file grezzi.

In [2]:
REBUILD_DERIVED = False

def load_od(path):
    return pd.read_csv(path, sep=';', header=[0, 1], index_col=[0, 1, 2]).select_dtypes(include='number')

def concentration_metrics(matrix):
    values = matrix.to_numpy().ravel()
    values = values[values > 0]
    ordered = np.sort(values)
    cumulative = np.cumsum(ordered)
    n = len(ordered)
    return {
        'total_flow': float(ordered.sum()),
        'n_positive': int(n),
        'gini': float((n + 1 - 2 * cumulative.sum() / cumulative[-1]) / n),
        'p90_share': float(ordered[ordered >= np.quantile(ordered, .90)].sum() / ordered.sum()),
        'p99_share': float(ordered[ordered >= np.quantile(ordered, .99)].sum() / ordered.sum()),
    }

if REBUILD_DERIVED:
    rows = []
    for filename, period in [('od-average-weekdays.csv', 'Weekdays'), ('od-average-holidays.csv', 'Holidays')]:
        metrics = concentration_metrics(load_od(RAW / filename))
        metrics.update({'dataset': filename, 'period': period})
        rows.append(metrics)
    structure = pd.DataFrame(rows)
else:
    structure = pd.read_csv(DER / 'od_structure_metrics.csv')
    structure['period'] = structure['dataset'].map({'od-average-weekdays.csv': 'Weekdays', 'od-average-holidays.csv': 'Holidays'})
structure[['period', 'total_flow', 'gini', 'p90_share', 'p99_share']]

,period,total_flow,gini,p90_share,p99_share
0,Weekdays,2665186.0,0.788047,0.708000,0.447600
1,Holidays,2198516.0,0.777042,0.689741,0.431929


In [3]:
fig, axes = plt.subplots(1, 2, figsize=(10.2, 4.0), constrained_layout=True)
sns.barplot(data=structure, x='period', y='total_flow', hue='period', legend=False, palette=COLORS, ax=axes[0])
axes[0].set(title='Intensita della domanda OD', xlabel='', ylabel='Flusso OD totale')
metrics_long = structure.melt(id_vars='period', value_vars=['gini', 'p90_share', 'p99_share'], var_name='metric', value_name='value')
labels = {'gini': 'Gini', 'p90_share': 'Quota top 10%', 'p99_share': 'Quota top 1%'}
metrics_long['metric'] = metrics_long['metric'].map(labels)
sns.barplot(data=metrics_long, x='metric', y='value', hue='period', palette=COLORS, ax=axes[1])
axes[1].set(title='Concentrazione della matrice OD', xlabel='', ylabel='Quota / indice', ylim=(0, 1.02))
axes[1].tick_params(axis='x', rotation=18)
axes[1].legend(title='', frameon=True)
save_figure(fig, 'od_baseline_and_concentration')
structure.to_csv(OUT / 'od_structure_summary.csv', index=False)

In [4]:
activity_path = RAW / 'city_activity.csv'
activity = pd.read_csv(activity_path, sep=';').sort_values('mean_points', ascending=False).head(15)
name_col = next((c for c in ['city', 'comune', 'name', 'location'] if c in activity.columns), activity.columns[0])
fig, ax = plt.subplots(figsize=(8.0, 5.0), constrained_layout=True)
sns.barplot(data=activity, y=name_col, x='mean_points', color='#146C94', ax=ax)
ax.set(title='Centri con maggiore attivita mobile media', xlabel='Punti attivi medi', ylabel='')
save_figure(fig, 'mobile_activity_top15')
print('Figure create:', sorted(p.name for p in OUT.glob('*.pdf')))

Figure create: ['mobile_activity_top15.pdf', 'od_baseline_and_concentration.pdf']
